In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb
import warnings
import shap
warnings.filterwarnings("ignore")

OSError: Could not find/load shared object file 'llvmlite.dll' from resource location: 'llvmlite.binding'. This could mean that the library literally cannot be found, but may also mean that the permissions are incorrect or that a dependency of/a symbol in the library could not be resolved.

In [2]:
#pip uninstall llvmlite numba -y
!pip install llvmlite numba

In [37]:
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    TF_AVAILABLE = True
except ImportError:
    TF_AVAILABLE = False
    print("TensorFlow not installed. LSTM branch will be skipped.")

In [38]:
merged_df = pd.read_csv("merged_df.csv")

In [39]:
#Configuration
# AQI thresholds — CPCB India standard
AQI_WORK_THRESHOLD = 200          # Above this → no work
FORECAST_DAYS      = 7
LOOKBACK_HOURS     = 168          # 7 days of hourly history for LSTM
 
# Columns expected in merged dataset
SENSOR_COLS  = ["pm25", "pm10", "temp", "humidity"]          # from sensor nodes
WEATHER_COLS = ["NO2", "Ozone", "CO", "RH", "WS", "WD", "SR", "RF", "AT"]
TARGET_COL   = "pm25"                                         # primary AQI driver

Feature Engineering

In [40]:
def compute_aqi(pm25: float) -> float:
    """CPCB AQI sub-index for PM2.5 (µg/m³ → AQI)."""
    breakpoints = [
        (0,    30,   0,   50),
        (31,   60,   51,  100),
        (61,   90,   101, 200),
        (91,   120,  201, 300),
        (121,  250,  301, 400),
        (251,  500,  401, 500),
    ]
    for (c_low, c_high, i_low, i_high) in breakpoints:
        if c_low <= pm25 <= c_high:
            return ((i_high - i_low) / (c_high - c_low)) * (pm25 - c_low) + i_low
    return 500.0 if pm25 > 500 else 0.0
 
 
def add_aqi_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Add AQI column and binary work label."""
    df = df.copy()
    df["aqi"] = df["pm25"].apply(compute_aqi)
    df["work_allowed"] = (df["aqi"] <= AQI_WORK_THRESHOLD).astype(int)
    return df
 
 
def build_lag_features(df: pd.DataFrame, target: str = TARGET_COL) -> pd.DataFrame:
    """
    Add lag and rolling-window features for XGBoost.
    All sensor and weather columns get lags; rolling stats on PM2.5.
    """
    df = df.copy()
    all_feat_cols = SENSOR_COLS + WEATHER_COLS
 
    for col in all_feat_cols:
        if col not in df.columns:
            continue
        for lag in [1, 3, 6, 12, 24, 48, 168]:           # 1h → 7d lags
            df[f"{col}_lag{lag}"] = df[col].shift(lag)
 
    # Rolling stats on primary target
    for window in [6, 24, 72, 168]:
        df[f"{target}_roll_mean_{window}"] = df[target].shift(1).rolling(window).mean()
        df[f"{target}_roll_std_{window}"]  = df[target].shift(1).rolling(window).std()
        df[f"{target}_roll_max_{window}"]  = df[target].shift(1).rolling(window).max()
 
    # Time-of-day and day-of-week (construction patterns)
    df["hour"]       = df.index.hour
    df["dayofweek"]  = df.index.dayofweek
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["month"]      = df.index.month
 
    # Wind-direction cyclical encoding
    if "WD" in df.columns:
        df["WD_sin"] = np.sin(np.deg2rad(df["WD"]))
        df["WD_cos"] = np.cos(np.deg2rad(df["WD"]))
 
    return df
 
 
def preprocess(df: pd.DataFrame, freq: str = "1h") -> pd.DataFrame:
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("DataFrame must have a DatetimeIndex.")

    # ── Separate object/categorical columns before any numeric ops ─────────────
    obj_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    df_obj   = df[obj_cols].copy() if obj_cols else pd.DataFrame(index=df.index)
    df_num   = df.drop(columns=obj_cols)

    # Coerce everything remaining to numeric (bad strings → NaN)
    df_num = df_num.apply(pd.to_numeric, errors="coerce")
    df_num = df_num.dropna(axis=1, how="all")

    # ── Resample numeric data only ─────────────────────────────────────────────
    df_num = df_num.resample(freq).mean()
    df_num = df_num.interpolate(method="time", limit=6)
    df_num = df_num.ffill(limit=3).bfill(limit=3)

    # ── Re-attach object columns via mode per period ───────────────────────────
    for col in obj_cols:
        df_num[col] = (
            df_obj[col]
            .resample(freq)
            .agg(lambda x: x.mode().iloc[0] if not x.empty and not x.mode().empty else np.nan)
        )

    df_num = add_aqi_labels(df_num)
    df_num = build_lag_features(df_num)
    df_num = df_num.dropna()

    return df_num

LSTM Model

In [41]:
def build_lstm_sequences(series: np.ndarray, lookback: int, horizon: int):
    """Create overlapping (X, y) windows for LSTM training."""
    X, y = [], []
    for i in range(lookback, len(series) - horizon + 1):
        X.append(series[i - lookback:i])
        y.append(series[i:i + horizon])
    return np.array(X), np.array(y)
 
 
class LSTMForecaster:
    """
    Bidirectional LSTM for 7-day PM2.5 forecasting.
    Uncertainty estimated via MC Dropout (forward pass N times at inference).
    """
 
    def __init__(self, lookback: int = LOOKBACK_HOURS,
                 horizon: int = FORECAST_DAYS * 24,
                 n_features: int = 1, mc_samples: int = 50):
        self.lookback    = lookback
        self.horizon     = horizon
        self.n_features  = n_features
        self.mc_samples  = mc_samples
        self.scaler      = MinMaxScaler()
        self.model       = None
 
    def _architecture(self) -> "tf.keras.Model":
        model = Sequential([
            Bidirectional(LSTM(128, return_sequences=True),
                          input_shape=(self.lookback, self.n_features)),
            Dropout(0.3),
            LSTM(64, return_sequences=False),
            Dropout(0.3),
            Dense(64, activation="relu"),
            Dropout(0.2),
            Dense(self.horizon),
        ])
        model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                      loss="huber",
                      metrics=["mae"])
        return model
 
    def fit(self, df: pd.DataFrame, feature_cols: list[str],
            epochs: int = 60, batch_size: int = 32,
            val_split: float = 0.15) -> dict:
        """Train LSTM; returns training history metrics."""
        arr = df[feature_cols].values
        arr_scaled = self.scaler.fit_transform(arr)
 
        X, y = build_lstm_sequences(arr_scaled, self.lookback, self.horizon)
        # y uses only the first column (pm25) as target
        y = y[:, :, 0] if y.ndim == 3 else y
 
        split = int(len(X) * (1 - val_split))
        X_tr, X_val = X[:split], X[split:]
        y_tr, y_val = y[:split], y[split:]
 
        self.model = self._architecture()
        callbacks = [
            EarlyStopping(patience=8, restore_best_weights=True),
            ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-6),
        ]
        history = self.model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=epochs, batch_size=batch_size,
            callbacks=callbacks, verbose=0,
        )
        return {
            "val_mae":  min(history.history["val_mae"]),
            "val_loss": min(history.history["val_loss"]),
        }
 
    def predict_with_uncertainty(self, X_input: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """
        MC Dropout inference.
        Returns (mean_forecast, std_forecast) in original units.
        """
        if self.model is None:
            raise RuntimeError("Model not trained yet.")
 
        # Enable dropout at inference time
        preds = np.stack([
            self.model(X_input, training=True).numpy()
            for _ in range(self.mc_samples)
        ], axis=0)                           # (mc_samples, batch, horizon)
 
        mean_pred = preds.mean(axis=0)
        std_pred  = preds.std(axis=0)
 
        # Inverse-scale (only first feature dimension)
        dummy = np.zeros((mean_pred.shape[1], self.scaler.n_features_in_))
        dummy[:, 0] = mean_pred[0]
        mean_unscaled = self.scaler.inverse_transform(dummy)[:, 0]
 
        dummy[:, 0] = std_pred[0]
        std_unscaled = self.scaler.inverse_transform(dummy)[:, 0]
 
        return mean_unscaled, std_unscaled
 
    def forecast_next_7_days(self, df: pd.DataFrame,
                              feature_cols: list[str]) -> pd.DataFrame:
        """
        Given recent data, return a DataFrame with:
          date, predicted_pm25, lower_ci, upper_ci, aqi, work_allowed, confidence
        """
        arr = df[feature_cols].values
        arr_scaled = self.scaler.transform(arr)
 
        window = arr_scaled[-self.lookback:][np.newaxis, ...]   # (1, lookback, feats)
        mean_pm25, std_pm25 = self.predict_with_uncertainty(window)
 
        # Hourly → daily aggregation
        n_hours = FORECAST_DAYS * 24
        mean_pm25 = mean_pm25[:n_hours]
        std_pm25  = std_pm25[:n_hours]
 
        daily_mean = mean_pm25.reshape(FORECAST_DAYS, 24).mean(axis=1)
        daily_std  = std_pm25.reshape(FORECAST_DAYS, 24).mean(axis=1)
 
        future_dates = pd.date_range(
            df.index[-1] + pd.Timedelta("1D"), periods=FORECAST_DAYS, freq="D"
        )
        out = pd.DataFrame({
            "date":          future_dates,
            "predicted_pm25": daily_mean,
            "lower_ci":      daily_mean - 1.96 * daily_std,
            "upper_ci":      daily_mean + 1.96 * daily_std,
        })
        out["aqi"]          = out["predicted_pm25"].apply(compute_aqi)
        out["work_allowed"] = (out["aqi"] <= AQI_WORK_THRESHOLD).astype(int)
        out["confidence"]   = 1 - (daily_std / (daily_mean + 1e-6)).clip(0, 1)
        return out

XGBoost Model

In [42]:
class XGBoostForecaster:
    """
    Direct multi-step XGBoost (one model per forecast day).
    SHAP values used for feature importance and change attribution.
    """
 
    def __init__(self, horizon: int = FORECAST_DAYS):
        self.horizon  = horizon
        self.models   = []
        self.explainer = None
        self.feature_cols = None
 
    def fit(self, df: pd.DataFrame, feature_cols: list[str],
            target: str = "pm25") -> dict:
        """Train one XGBoost model per forecast step (DIRMO strategy)."""
        self.feature_cols = feature_cols
        X = df[feature_cols]
        metrics = {}
 
        tscv = TimeSeriesSplit(n_splits=5)
 
        for step in range(1, self.horizon + 1):
            y = df[target].shift(-step * 24)    # step days ahead (daily)
            mask = y.notna()
            X_s, y_s = X[mask], y[mask]
 
            rmse_scores = []
            for tr_idx, val_idx in tscv.split(X_s):
                model = xgb.XGBRegressor(
                    n_estimators=500,
                    learning_rate=0.05,
                    max_depth=6,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    min_child_weight=3,
                    reg_alpha=0.1,
                    reg_lambda=1.0,
                    tree_method="hist",
                    early_stopping_rounds=20,
                    verbosity=0,
                )
                model.fit(
                    X_s.iloc[tr_idx], y_s.iloc[tr_idx],
                    eval_set=[(X_s.iloc[val_idx], y_s.iloc[val_idx])],
                    verbose=False,
                )
                preds = model.predict(X_s.iloc[val_idx])
                rmse_scores.append(
                    np.sqrt(mean_squared_error(y_s.iloc[val_idx], preds))
                )
 
            # Final fit on all data
            final_model = xgb.XGBRegressor(
                n_estimators=500, learning_rate=0.05, max_depth=6,
                subsample=0.8, colsample_bytree=0.8,
                min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
                tree_method="hist", verbosity=0,
            )
            final_model.fit(X_s, y_s)
            self.models.append(final_model)
            metrics[f"day{step}_rmse"] = np.mean(rmse_scores)
 
        # Build SHAP explainer on the Day-1 model
        self.explainer = shap.TreeExplainer(self.models[0])
        return metrics
 
    def forecast_next_7_days(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return 7-day forecast with confidence proxy (prediction interval via quantile boost)."""
        X_latest = df[self.feature_cols].iloc[[-1]]
 
        predictions, shap_vals_list = [], []
        for model in self.models:
            pred = model.predict(X_latest)[0]
            predictions.append(max(pred, 0))
            sv = self.explainer.shap_values(X_latest)
            shap_vals_list.append(sv[0])
 
        mean_pm25 = np.array(predictions)
 
        # Confidence proxy: higher RMSE CV → lower confidence
        future_dates = pd.date_range(
            df.index[-1] + pd.Timedelta("1D"), periods=self.horizon, freq="D"
        )
        out = pd.DataFrame({
            "date":           future_dates,
            "predicted_pm25": mean_pm25,
            "lower_ci":       mean_pm25 * 0.85,
            "upper_ci":       mean_pm25 * 1.15,
        })
        out["aqi"]          = out["predicted_pm25"].apply(compute_aqi)
        out["work_allowed"] = (out["aqi"] <= AQI_WORK_THRESHOLD).astype(int)
        out["confidence"]   = 1 - (mean_pm25 / 500).clip(0, 1)
 
        return out, np.array(shap_vals_list)
 
    def top_drivers(self, shap_matrix: np.ndarray, top_n: int = 10) -> pd.DataFrame:
        """Return top N features driving change across the 7-day horizon."""
        mean_abs_shap = np.abs(shap_matrix).mean(axis=0)
        idx_sorted    = np.argsort(mean_abs_shap)[::-1][:top_n]
        return pd.DataFrame({
            "feature":       [self.feature_cols[i] for i in idx_sorted],
            "mean_abs_shap": mean_abs_shap[idx_sorted],
        })

CHANGE ATTRIBUTION  (previous day → next day driver)

In [43]:
def explain_change_between_days(
    df: pd.DataFrame,
    xgb_forecaster: XGBoostForecaster,
    date_from: str,
    date_to: str,
) -> pd.DataFrame:
    """
    Compute delta SHAP to explain what caused the PM2.5 change
    from one date to another.
 
    Returns a DataFrame of features sorted by their contribution to the change.
    """
    fc = xgb_forecaster.feature_cols
    row_from = df.loc[date_from:date_from, fc].iloc[[0]]
    row_to   = df.loc[date_to:date_to, fc].iloc[[0]]
 
    exp = xgb_forecaster.explainer
    sv_from = exp.shap_values(row_from)
    sv_to   = exp.shap_values(row_to)
 
    delta_shap = sv_to[0] - sv_from[0]
 
    result = pd.DataFrame({
        "feature":     fc,
        "delta_shap":  delta_shap,
        "direction":   np.where(delta_shap > 0, "↑ increased AQI", "↓ decreased AQI"),
    }).sort_values("delta_shap", key=abs, ascending=False)
 
    return result

SENSOR ATTRIBUTION SUBTASK
#     One CPCB station + N co-located sensor sites
#     → rank which site drives pollution

In [44]:
def attribute_pollution_to_sites(
    merged_df: pd.DataFrame,
    target:    str = "pm25",
    bg_col:    str = "pm25_cpcb",          # CPCB background PM2.5 column
) -> pd.DataFrame:
    """
    For each sensor site, compute:
      - delta_pm25  : site PM2.5 minus CPCB background (construction contribution)
      - contribution_pct : site's share of total excess PM2.5
      - mean_aqi     : time-averaged AQI from that site
 
    Args:
        merged_df : DataFrame with columns [site_col, target, bg_col, ...]
                    Must contain one row per (timestamp, site) after wide-to-long pivot.
        bg_col    : column name for the CPCB background PM2.5 reading.
 
    Returns:
        DataFrame ranked by pollution contribution.
    """
    if bg_col not in merged_df.columns:
        # If CPCB background is not a separate column, use the dataset mean as proxy
        bg_mean = merged_df[target].mean()
        merged_df[bg_col] = bg_mean
 
    site_stats = (
        merged_df
        .groupby(site_col)
        .agg(
            mean_pm25    = (target, "mean"),
            std_pm25     = (target, "std"),
            mean_bg      = (bg_col, "mean"),
            n_readings   = (target, "count"),
        )
        .reset_index()
    )
 
    site_stats["delta_pm25"]  = (site_stats["mean_pm25"] - site_stats["mean_bg"]).clip(lower=0)
    total_excess               = site_stats["delta_pm25"].sum()
    site_stats["contribution_pct"] = (
        (site_stats["delta_pm25"] / (total_excess + 1e-9)) * 100
    ).round(2)
    site_stats["mean_aqi"]    = site_stats["mean_pm25"].apply(compute_aqi).round(1)
    site_stats["risk_label"]  = pd.cut(
        site_stats["mean_aqi"],
        bins=[0, 50, 100, 200, 300, 500],
        labels=["Good", "Satisfactory", "Moderate", "Poor", "Very Poor"],
    )
 
    return site_stats.sort_values("contribution_pct", ascending=False).reset_index(drop=True)
 
 
def run_xgb_attribution_per_site(
    merged_df:    pd.DataFrame,
    feature_cols: list[str] = None,
    target:       str = "pm25",
) -> dict[str, pd.DataFrame]:
    """
    Train a separate XGBoost model per sensor site and compute SHAP-based
    feature importance. Useful to understand *why* a specific site pollutes more
    (e.g., wind direction from a construction hotspot, temperature inversion).
 
    Returns a dict: {site_id: top_drivers_dataframe}
    """
    if feature_cols is None:
        feature_cols = WEATHER_COLS + ["hour", "dayofweek", "is_weekend"]
 
    site_results = {}
    for site_id, group in merged_df.groupby(site_col):
        group = group.dropna(subset=feature_cols + [target])
        if len(group) < 100:
            continue
 
        X, y = group[feature_cols], group[target]
        model = xgb.XGBRegressor(
            n_estimators=300, max_depth=5, learning_rate=0.05,
            subsample=0.8, tree_method="hist", verbosity=0,
        )
        model.fit(X, y)
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X)
        mean_abs = np.abs(sv).mean(axis=0)
 
        site_results[str(site_id)] = pd.DataFrame({
            "feature":       feature_cols,
            "mean_abs_shap": mean_abs,
        }).sort_values("mean_abs_shap", ascending=False).head(10)
 
    return site_results

MODEL SELECTION — compare LSTM vs XGBoost confidence

In [45]:
def select_best_model(
    lstm_forecast:  pd.DataFrame | None,
    xgb_forecast:   pd.DataFrame,
    lstm_val_mae:   float | None = None,
    xgb_cv_rmse:    float | None = None,
) -> tuple[str, pd.DataFrame]:
    """
    Selects the more confident model based on:
      1. Average prediction confidence score
      2. Validation error (MAE / RMSE)
      3. Width of confidence interval (narrower = more certain)
 
    Returns (model_name, chosen_forecast_df)
    """
    scores = {}
 
    if lstm_forecast is not None:
        ci_width_lstm = (lstm_forecast["upper_ci"] - lstm_forecast["lower_ci"]).mean()
        lstm_score = (
            lstm_forecast["confidence"].mean()
            - (ci_width_lstm / 500)                       # penalise wide CI
            - (lstm_val_mae / 500 if lstm_val_mae else 0)
        )
        scores["LSTM"] = (lstm_score, lstm_forecast)
 
    ci_width_xgb = (xgb_forecast["upper_ci"] - xgb_forecast["lower_ci"]).mean()
    xgb_score = (
        xgb_forecast["confidence"].mean()
        - (ci_width_xgb / 500)
        - (xgb_cv_rmse / 500 if xgb_cv_rmse else 0)
    )
    scores["XGBoost"] = (xgb_score, xgb_forecast)
 
    best = max(scores, key=lambda k: scores[k][0])
    print(f"\n=== Model Selection ===")
    for name, (score, _) in scores.items():
        print(f"  {name:10s}: composite score = {score:.4f}")
    print(f"  → Selected: {best}\n")
 
    return best, scores[best][1]

Complete Pipelines

In [49]:
def run_pipeline(df_raw: pd.DataFrame) -> dict:
    """
    End-to-end pipeline.
 
    Args:
        df_raw : merged sensor + weather DataFrame with DatetimeIndex.
                 Must contain at minimum: pm25, pm10, temp, humidity,
                 NO2, Ozone, CO, RH, WS, WD, SR, RF, AT.
                 Optional: site_id column for multi-site attribution.
 
    Returns dict with keys:
        forecast        : chosen 7-day forecast DataFrame
        best_model      : "LSTM" or "XGBoost"
        site_attribution: ranked pollution source DataFrame
        change_drivers  : top drivers of yesterday→today change
        site_shap_dict  : per-site SHAP driver breakdown
    """
    print("── Step 1: Preprocessing ──")
    df = preprocess(df_raw)
    print(f"   Processed shape: {df.shape}")
 
    # Feature columns for XGBoost — numeric only, exclude site_id and labels
    lag_cols  = [c for c in df.columns if "_lag" in c or "_roll_" in c]
    time_cols = ["hour", "dayofweek", "is_weekend", "month", "WD_sin", "WD_cos"]
    base_cols = [c for c in SENSOR_COLS + WEATHER_COLS if c in df.columns]
    exclude   = {"aqi", "work_allowed", "pm25_cpcb"}
    xgb_features = [
        c for c in base_cols + lag_cols + time_cols
        if c in df.columns
        and c not in exclude
        and pd.api.types.is_numeric_dtype(df[c])
    ]
 
    # ── XGBoost ────────────────────────────────────────────────────────────────
    print("── Step 2: Training XGBoost ──")
    xgb_fc = XGBoostForecaster(horizon=FORECAST_DAYS)
    xgb_metrics = xgb_fc.fit(df, xgb_features, target="pm25")
    day1_rmse = xgb_metrics.get("day1_rmse", None)
    print(f"   XGBoost Day-1 CV RMSE: {day1_rmse:.2f}" if day1_rmse else "")
    xgb_forecast, shap_matrix = xgb_fc.forecast_next_7_days(df)
 
    # ── LSTM ───────────────────────────────────────────────────────────────────
    lstm_forecast, lstm_mae = None, None
    if TF_AVAILABLE:
        print("── Step 3: Training LSTM ──")
        lstm_features = base_cols
        lstm_fc = LSTMForecaster(
            lookback=LOOKBACK_HOURS,
            horizon=FORECAST_DAYS * 24,
            n_features=len(lstm_features),
        )
        hist = lstm_fc.fit(df, lstm_features, epochs=60)
        lstm_mae = hist["val_mae"]
        print(f"   LSTM val MAE: {lstm_mae:.3f}")
        lstm_forecast = lstm_fc.forecast_next_7_days(df, lstm_features)
    else:
        print("── Step 3: LSTM skipped (TF not installed) ──")
 
    # ── Model selection ────────────────────────────────────────────────────────
    print("── Step 4: Model selection ──")
    best_name, best_forecast = select_best_model(
        lstm_forecast, xgb_forecast,
        lstm_val_mae=lstm_mae, xgb_cv_rmse=day1_rmse,
    )
 
    # ── Change attribution (yesterday → today) ─────────────────────────────────
    print("── Step 5: Change attribution ──")
    yesterday = (df.index[-1] - pd.Timedelta("1D")).strftime("%Y-%m-%d")
    today     = df.index[-1].strftime("%Y-%m-%d")
    try:
        change_drivers = explain_change_between_days(df, xgb_fc, yesterday, today)
    except Exception:
        change_drivers = pd.DataFrame()
 
    # ── Sensor site attribution ────────────────────────────────────────────────
    print("── Step 6: Sensor attribution ──")
    site_attr     = pd.DataFrame()
    site_shap_dict = {}
 
    if SITE_COLUMN in df.columns:
        site_attr = attribute_pollution_to_sites(df, site_col=SITE_COLUMN)
        site_shap_dict = run_xgb_attribution_per_site(
            df, site_col=SITE_COLUMN,
            feature_cols=[c for c in WEATHER_COLS + time_cols
                          if c in df.columns
                          and pd.api.types.is_numeric_dtype(df[c])],
        )
        print(site_attr[["site_id", "mean_pm25", "delta_pm25",
                           "contribution_pct", "risk_label"]].to_string())
    else:
        print("   No 'site_id' column found — skipping sensor attribution.")
 
    # ── Print 7-day forecast ───────────────────────────────────────────────────
    print("\n── 7-Day Forecast ──")
    print(best_forecast[["date", "predicted_pm25", "aqi",
                          "work_allowed", "confidence"]].to_string(index=False))
 
    return {
        "forecast":         best_forecast,
        "best_model":       best_name,
        "xgb_top_drivers":  xgb_fc.top_drivers(shap_matrix),
        "change_drivers":   change_drivers,
        "site_attribution": site_attr,
        "site_shap_dict":   site_shap_dict,
    }

Quick Demo

In [50]:
if __name__ == "__main__":
    np.random.seed(42)
    n = 8760    # 1 year of hourly data
 
    dates = pd.date_range("2023-01-01", periods=n, freq="1h")
    #sites = np.random.choice(["Site_A", "Site_B", "Site_C", "Site_D"], size=n)
 
    df_demo = pd.DataFrame({
        "pm25":     np.clip(50 + 30 * np.sin(np.linspace(0, 6*np.pi, n))
                            + np.random.normal(0, 15, n), 5, 400),
        "pm10":     np.clip(80 + 40 * np.sin(np.linspace(0, 6*np.pi, n))
                            + np.random.normal(0, 20, n), 10, 600),
        "temp":     25 + 8 * np.sin(np.linspace(0, 2*np.pi, n))
                    + np.random.normal(0, 2, n),
        "humidity": np.clip(60 + 20 * np.cos(np.linspace(0, 2*np.pi, n))
                            + np.random.normal(0, 5, n), 10, 100),
        "NO2":      np.clip(30 + np.random.normal(0, 8, n), 0, 200),
        "Ozone":    np.clip(40 + np.random.normal(0, 10, n), 0, 180),
        "CO":       np.clip(0.8 + np.random.normal(0, 0.2, n), 0, 10),
        "RH":       np.clip(62 + np.random.normal(0, 6, n), 0, 100),
        "WS":       np.clip(3 + np.random.exponential(2, n), 0, 20),
        "WD":       np.random.uniform(0, 360, n),
        "SR":       np.clip(300 * np.abs(np.sin(np.linspace(0, n*np.pi/12, n)))
                            + np.random.normal(0, 30, n), 0, 1000),
        "RF":       np.clip(np.random.exponential(0.5, n), 0, 50),
        "AT":       25 + 8 * np.sin(np.linspace(0, 2*np.pi, n))
                    + np.random.normal(0, 1.5, n),
        #"site_id":  sites,
    }, index=dates)
 
    results = run_pipeline(df_demo)
 
    print("\n── Top XGBoost Feature Drivers ──")
    print(results["xgb_top_drivers"].to_string(index=False))
 
    if not results["change_drivers"].empty:
        print("\n── Yesterday → Today Change Drivers ──")
        print(results["change_drivers"].head(8).to_string(index=False))
 
    if not results["site_attribution"].empty:
        print("\n── Sensor Site Pollution Attribution ──")
        print(results["site_attribution"].to_string(index=False))
 

── Step 1: Preprocessing ──
   Processed shape: (8592, 124)
── Step 2: Training XGBoost ──


NameError: name 'shap' is not defined